# Test a mesh-transformer checkpoint

Loads the **same** splits a training run used -- same config, same seed, same
`random_split` -- generates LOD2 from each LOD1 condition, and shows
input / ground truth / generated side by side for `train`, `val` and `test`.

The train split is a memorization check, not a score: those buildings are the
ones the weights were fit on, so the gap between train and val/test is the
overfitting readout, and a train building that still comes out wrong is a
capacity or optimization problem rather than a generalization one.

The **coordinate** path only: nine tokens per triangle, and `detokenize` is the
exact inverse of `tokenize`, so nothing between the model and the mesh can lose
anything. `notebooks/test-mesh.ipynb` is the sibling for the other tokenizer.

Generation has no KV cache: one forward pass per token, so keep `N_SHOW` small --
it is now per split, so the cost is three times what the same number used to be.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.dataset.mesh_datamodule import MeshDataModule
from src.dataset.mesh_dataset import (amt_detokenize, detokenize,
                                      mesh_collate_fn, specials)
from src.eval.mesh_metrics import mesh_metrics
from src.models.mesh_transformer import MeshTransformerModule
from src.utils.initialization import load_config
from src.visualize_mesh import cityjson_figure_from_mesh, mesh_figure, side_by_side

import plotly.io as pio

pio.renderers.default = "notebook" 

## The experiment to test

`CKPT = None` picks the newest checkpoint of the run the config names.

In [ ]:
CONFIG = ROOT / "configs" / "mesh2-train.yaml"
CKPT = None          # None = newest .ckpt under the run directory
SPLITS = ("train", "val", "test")   # which splits to generate from
N_SHOW = 4           # buildings PER SPLIT; each one is slow
TEMPERATURE = 0.0    # 0 = argmax, as in mesh_eval

cfg = load_config(CONFIG, [])

# Everything below decodes with `detokenize`, which only speaks coordinates. A
# run that spells tokens some other way would load here and generate silently
# meaningless geometry, so refuse it at the top instead.
if cfg.mesh_model.tokenizer != "coord":
    raise ValueError(
        f"{CONFIG.name} is a '{cfg.mesh_model.tokenizer}' run -- this notebook "
        "is the coordinate path. Use notebooks/test-mesh.ipynb.")

run_dir = ROOT / cfg.logging.save_dir / cfg.logging.experiment_name / cfg.logging.run_name

if CKPT is None:
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt under {run_dir} -- set CKPT explicitly.")
    CKPT = ckpts[-1]

# The dataset spells tokens the way `mesh_data.tokenization` says; the model
# decodes them the way its own saved `tokenization` hyperparameter says. Nothing
# downstream reconciles the two, and a mismatch does NOT raise -- it silently
# decodes with the wrong inverse. A coordinate sequence read as AMT keeps the
# building's vertices and silhouette exactly, and re-triangulates them into
# ~1.8x as many faces (64 -> 116 measured), which reads as "AMT is broken"
# rather than as a config error. Hence the check.
_ckpt_tok = torch.load(CKPT, map_location="cpu",
                       weights_only=False)["hyper_parameters"].get("tokenization")
if _ckpt_tok != cfg.mesh_data.tokenization:
    raise ValueError(
        f"{Path(CKPT).name} was trained with tokenization={_ckpt_tok!r}, but "
        f"{CONFIG.name} sets mesh_data.tokenization="
        f"{cfg.mesh_data.tokenization!r}. Point CONFIG at the config this "
        "checkpoint was trained with -- decoding one with the other's inverse "
        "produces a plausible-looking but wrong mesh.")

print(f"config {CONFIG.name}, run {cfg.logging.run_name}, "
      f"tokenization {cfg.mesh_data.tokenization}")
print(f"checkpoint {Path(CKPT).relative_to(ROOT)}")

## The same splits

Every argument below is what `src/train_mesh.py` passes, `cfg.seed` included, so
`random_split` reproduces the exact partition the run trained against -- which is
what makes the train split a train split and not a random subset.

In [ ]:
datamodule = MeshDataModule(
    dataset_dir=ROOT / cfg.mesh_data.dataset_dir,
    lod_in=cfg.mesh_data.lod_in,
    lod_out=cfg.mesh_data.lod_out,
    num_bins=cfg.mesh_data.num_bins,
    margin_lo=list(cfg.mesh_data.margin_lo),
    margin_hi=list(cfg.mesh_data.margin_hi),
    max_faces=cfg.mesh_data.max_faces,
    max_files=cfg.mesh_data.max_files,
    batch_size=cfg.training.batch_size,
    train_val_test_split=tuple(cfg.training.train_val_test_split),
    num_workers=0,
    seed=cfg.seed,
    # NOT optional, and it defaults to "coord" if you leave it out. The dataset
    # spells tokens this way and `decode_tokens` reads them back the way the
    # CHECKPOINT was trained -- omitting it here handed an AMT model coordinate
    # tokens, which it re-triangulated into ~1.8x the faces (56 -> 103 measured
    # on the LOD1 condition, 64 -> 117 on the target). Same vertices, same
    # silhouette, extra triangles fanned across the interior.
    tokenization=cfg.mesh_data.tokenization,
)
datamodule.setup()

# The same `Subset` objects the dataloaders wrap, so position i here is the same
# building the run saw at position i of that split.
splits = {name: getattr(datamodule, f"{name}_dataset") for name in SPLITS}

print(f"train={len(datamodule.train_dataset)} val={len(datamodule.val_dataset)} "
      f"test={len(datamodule.test_dataset)}")
print("generating from: " + ", ".join(f"{n} ({min(N_SHOW, len(s))})"
                                      for n, s in splits.items()))

In [ ]:
# Nothing to rebuild and hand in: the coordinate tokenizer is a pair of pure
# functions, not a module, so the checkpoint is the whole model.
model = MeshTransformerModule.load_from_checkpoint(CKPT, map_location="cpu")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Same budget rule as MeshEvalCallback -- 9 tokens per triangle plus the stop
# token -- and asking past the positional embedding raises.
per_face = 9
max_new_tokens = min(per_face * (cfg.mesh_data.max_faces or 200) + 1,
                     model.network.max_seq_len - 1)
print(f"{model.num_bins} bins, {per_face} tokens/face, "
      f"max_seq_len {model.network.max_seq_len}, budget {max_new_tokens} tokens on {device}")

## Generate

In [ ]:
def to_metres(mesh, center, scale):
    """(verts, faces) in the unit box -> metres, undoing the LOD1-box frame."""
    verts, faces = mesh
    return verts * scale + center, faces


results = []

for split, dataset in splits.items():
    items = [dataset[i] for i in range(min(N_SHOW, len(dataset)))]
    if not items:
        continue
    batch = mesh_collate_fn(items, pad=specials(model.num_bins)[2])
    batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

    with torch.no_grad():
        # `_prepare` is what the training step calls, so generation starts from
        # exactly the condition and target tensors the model was fit against.
        cond, tgt, cond_pad, _ = model._prepare(batch)
        out = model.generate(cond, cond_pad,
                             max_new_tokens=max_new_tokens, temperature=TEMPERATURE)

    for k, name in enumerate(batch["ids"]):
        center = batch["center"][k].cpu().numpy()
        scale = batch["scale"][k].cpu().numpy()
        # All three go through the same tokenizer, and `detokenize` is its exact
        # inverse, so what is left between ground truth and generation is model
        # error with no tokenizer loss underneath it.
        # The condition is spelled in the run's own tokenization, so it needs
        # the run's own inverse. Hard-coding `detokenize` here decoded an AMT
        # condition as coordinates and drew a mesh with half the faces.
        cond_inverse = amt_detokenize if model.tokenization == "amt" else detokenize
        lod1 = to_metres(cond_inverse(batch["cond"][k].cpu().numpy(), model.num_bins),
                         center, scale)
        gt = to_metres(model.decode_tokens(tgt[k]), center, scale)
        gen = to_metres(model.decode_tokens(out[k]), center, scale)
        results.append((split, name, lod1, gt, gen))
        print(f"{split}/{name}: lod1 {len(lod1[1])} tris | gt {len(gt[1])} | "
              f"gen {len(gen[1])}")

## Input / ground truth / generated

Top row: the raw triangle meshes. All three are the coordinate tokenizer's exact
inverse -- the LOD1 input is the condition, never decoded by the model, and the
round trip is lossless -- so what is left between the middle and right panel is
model error alone.

Bottom row: the same geometry through `mesh_to_cityjson` -- coplanar triangles merged
back into polygons and coloured by semantic surface (blue ground, orange roof, grey
wall). An empty bottom panel means the writer refused the mesh.

Every panel is labelled with the split it came from. Read the `train` rows first:
if generation is already wrong on buildings the weights were fit on, nothing about
`val` or `test` is a generalization finding.

In [ ]:
import plotly.graph_objects as go

figs = {}


def city_panel(mesh):
    """`cityjson_figure_from_mesh`, with a blank panel where it cannot run.

    A free-running generation can come back empty or as a soup the writer
    rejects, and across three splits that is common enough that letting it raise
    would cost every figure after it.
    """
    if not len(mesh[1]):
        return go.Figure(), 0
    try:
        return cityjson_figure_from_mesh(*mesh)
    except Exception as exc:
        print(f"  CityJSON write-back failed: {exc}")
        return go.Figure(), 0


for split, name, lod1, gt, gen in results:
    row = mesh_metrics(gen, gt, taus=tuple(cfg.mesh_eval.taus),
                       n_points=cfg.mesh_eval.n_points, voxel_m=cfg.mesh_eval.voxel_m)
    print(split, name, {k: round(v, 4) for k, v in sorted(row.items()) if np.isfinite(v)})

    meshes = [(lod1, "#898781"), (gt, "#2a78d6"), (gen, "#eda100")]
    top = [mesh_figure(*mesh, color=color) for mesh, color in meshes]
    bottom = [city_panel(mesh) for mesh, _ in meshes]

    figs[f"{split}/{name}"] = side_by_side(
        [top, [fig for fig, _ in bottom]],
        [f"[{split}] LOD1 input, {len(lod1[1])} tris",
         f"[{split}] LOD2 ground truth, {len(gt[1])} tris",
         f"[{split}] LOD2 generated, {len(gen[1])} tris"]
        + [f"CityJSON, {n} surfaces" if n else "CityJSON: none" for _, n in bottom],
    )

print(f"\n{len(figs)} figures built -- display them one at a time in the last cell.")

## Post-decode cleanup: what MeshAnything's post-processing would recover

`buaacyw/MeshAnything`'s `main.py` runs three trimesh calls on the decoded
triangle soup before it writes the `.obj`, and **nothing in this project's
pipeline runs any of them**:

| reference step | what it repairs | already covered here? |
|---|---|---|
| `merge_vertices()` | welds coincident vertices | yes, by `canonicalize` |
| `update_faces(unique_faces())` | drops duplicate faces | **no** |
| `fix_normals()` | repairs inconsistent winding | **no** |

`src.eval.mesh_postprocess` reproduces all three on numpy arrays, so no trimesh
dependency is added for a diagnostic. It is deliberately **not** wired into
`mesh_eval`, `MeshEvalCallback` or inference -- this panel measures what the
steps would buy, so that decision gets made on numbers rather than on the
reference's authority. See `docs/2026-08-12-mesh-tokenizer-path-issues.md`,
Issue 4.

`gt` here is the exact tokenizer inverse of the dataset target -- round tripping
it is provably lossless (0/200 mismatches measured) -- so the `gt` rows are a
control that should show *no change at all*. Any movement there is a bug in the
cleanup, not a finding.

The `gen` rows are the real measurement. The panel above already reports
`watertight_gen: 0.0` against `watertight_gt: 1.0` on every building, so
generation is losing watertightness that the representation had. Whether these
three steps recover it is exactly the question Issue 4 asks: if they do, wire
them into `mesh_eval`; if they do not, the gap is real geometry and Issue 4
closes.

Degenerate faces (two vertices collapsed onto one bin by `quantize`) are counted
but **not** removed, matching the reference: it calls `unique_faces`, not
`nondegenerate_faces`.

In [ ]:
import pandas as pd

from src.eval.mesh_postprocess import report

# Same reference as the panel above: `gt` *is* the exact tokenizer inverse of the
# dataset target, so it is what `gen` gets scored against. The `gt` rows are
# self-scored, hence no chamfer for them.
_metric = dict(taus=tuple(cfg.mesh_eval.taus), n_points=cfg.mesh_eval.n_points,
               voxel_m=cfg.mesh_eval.voxel_m)

rows = []
for split, name, lod1, gt, gen in results:
    for label, mesh in (("gt", gt), ("gen", gen)):
        r = report(mesh)
        chamfer = "-"
        if label == "gen" and len(mesh[1]):
            before = mesh_metrics(mesh, gt, **_metric)["chamfer_m"]
            after = mesh_metrics(r["mesh"], gt, **_metric)["chamfer_m"]
            chamfer = f"{before:.4f} -> {after:.4f}"
        rows.append({
            "split": split, "building": name, "mesh": label,
            "faces": f"{r['faces_before']} -> {r['faces_after']}",
            "verts": f"{r['verts_before']} -> {r['verts_after']}",
            "degen": f"{r['degenerate_before']} -> {r['degenerate_after']}",
            "watertight": f"{r['watertight_before']} -> {r['watertight_after']}",
            "chamfer_m": chamfer,
        })


def _moved(row, key):
    a, b = row[key].split(" -> ")
    return a != b


control = [r for r in rows if r["mesh"] == "gt"]
gen_rows = [r for r in rows if r["mesh"] == "gen"]

# The control must not move: gt is a lossless round trip, so cleanup has nothing
# to do. If this trips, suspect mesh_postprocess before suspecting the model.
# (Degenerate faces are exempt -- `quantize` creates those and the reference does
# not remove them either.)
bad = [r for r in control if _moved(r, "faces") or _moved(r, "verts")]
assert not bad, ("cleanup altered the lossless gt round trip on "
                 f"{[(r['split'], r['building']) for r in bad]}")

n_wt = sum(_moved(r, "watertight") for r in gen_rows)
n_face = sum(_moved(r, "faces") for r in gen_rows)
print(f"control ok: cleanup is a no-op on all {len(control)} gt meshes")
print(f"generated: watertight changed on {n_wt}/{len(gen_rows)}, "
      f"face count changed on {n_face}/{len(gen_rows)}")
if n_wt == n_face == 0:
    print("-> no-op on this sample: the watertight gap is geometry, and Issue 4 closes")
else:
    print("-> cleanup recovers something: candidate for wiring into mesh_eval (Issue 4)")
print("   N_SHOW buildings per split is a spot check, not a rate -- widen it "
      "before deciding")

# Split-major, so the train rows read as the memorization control they are.
pd.DataFrame(rows).set_index(["split", "building", "mesh"])

In [ ]:
# One at a time: each figure carries six 3-D scenes, and displaying all of them
# is what makes this notebook multi-megabyte on disk.
print(list(figs))

KEY = list(figs)[0]
figs[KEY]